# Solute transport convergence: MESS vs ESS

This notebook mirrors the mPCN/pCN convergence experiment structure, replacing:
- mPCN with MESS (M=100), and
- pCN with ESS (equivalently MESS with M=1).

Each replicate runs:
1. one MESS chain,
2. one group of P independent ESS chains.

Missing chains are generated; existing chains are reused.

In [ ]:
import sys
import json
import time
import hashlib
from pathlib import Path

import numpy as np

def resolve_repo_root(start: Path) -> Path:
    root = start.resolve()
    while root != root.parent and not (root / 'pyproject.toml').exists():
        root = root.parent
    return root

repo_root = resolve_repo_root(Path.cwd())
mess_src = repo_root / 'src'
mcmc_repo_root = repo_root.parent / 'mcmc-internal'
mcmc_src = mcmc_repo_root / 'src'

for p in [mess_src, mcmc_src]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from mess.problems.advection_diffusion import (
    AdvectionDiffusionToy,
    make_omegas_power,
    make_Astar_nn,
    make_Astar_from_atrue,
    params_from_skew,
    prior_diag_from_powerlaw,
    solve_theta,
)
from mess.algorithms.mess import mess_step

from multiproposal.diagnostics.observables import (
    make_parameter_observables,
    make_antisymmetric_A_observables,
    select_observables,
    observable_series,
)
from multiproposal.diagnostics.running_mse import (
    update_parameter_observables,
    add_first_row_sum_observable,
    update_A_observables,
    wrap_A_observables_for_params,
    load_cached_observable_targets,
    compute_single_observable_target,
    save_observable_target_cache,
    load_cached_mse_results,
    save_mse_cache,
)
from multiproposal.plotting.solute_transport_convergence import (
    plot_solute_transport_pub_traceplots,
    plot_solute_transport_running_mse,
)

print('repo_root:', repo_root)
print('mcmc_repo_root:', mcmc_repo_root)

In [ ]:
seed_data = 0
seed_mcmc = 202
config_id = 2

configurations = {
    1: {'d': 20, 'obs_highest_freq': 8, 'obs_bandwidth': 5},
    2: {'d': 40, 'obs_highest_freq': 12, 'obs_bandwidth': 7},
    3: {'d': 60, 'obs_highest_freq': 16, 'obs_bandwidth': 9},
    4: {'d': 40, 'obs_highest_freq': 8, 'obs_bandwidth': 5},
}
selected_config = configurations[config_id]
d = selected_config['d']
obs_highest_freq = selected_config['obs_highest_freq']
obs_bandwidth = selected_config['obs_bandwidth']

kappa = 0.02
sigma = 0.5
alpha = 3.0
gamma = 2.0
tau2 = 2.0
a_mode = 'nearest_neighbor'

P = 100
num_ess_chains = 100
num_mess_chains = 50
burn_in = 0
n_iters = 500

target_replicates = min(20, num_mess_chains)
mess_M = 100

# Posterior target configuration: prefer loading MPCN cached targets,
# otherwise estimate targets from a long MESS chain.
prefer_mpcn_target_cache = True
long_target_mess_M = 10
long_target_n_iters = 20000
long_target_burnin = 5000

true_observable_cache_mode = 'missing'
true_observable_force_ids = []
A_band_k = 2
A_top_k = 5
A_support_eps = 0.001
A_row_index = 0
A_use_full_row = False
trace_observable_ids = None
mse_observable_ids = [3, 102, 101, 8]
pub_plot_observable_ids = [3, 102, 101, 8]

rho_tag = f'messM{mess_M}'

print('d:', d, 'P:', P, 'target_replicates:', target_replicates, 'n_iters:', n_iters)
print('Target source: MPCN cache fallback to long MESS chain')

In [ ]:
def canonicalize_payload(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {k: canonicalize_payload(v) for k, v in sorted(obj.items())}
    if isinstance(obj, (list, tuple)):
        return [canonicalize_payload(v) for v in obj]
    return obj

def stable_hash(payload, length=12):
    data = json.dumps(canonicalize_payload(payload), sort_keys=True, separators=(',', ':'), ensure_ascii=True).encode('utf-8')
    return hashlib.sha256(data).hexdigest()[:length]

def get_obs_indices(dim_value, highest_freq, bandwidth):
    highest_freq = min(highest_freq, dim_value)
    bandwidth = min(bandwidth, dim_value)
    start = max(0, highest_freq - bandwidth + 1)
    return np.arange(start, highest_freq + 1, dtype=int)

def get_param_indices_for_dim(dim_value, shared_draws):
    cache = shared_draws.setdefault('param_indices_cache', {})
    if dim_value not in cache:
        iju = shared_draws['param_iju']
        mask = (iju[0] < dim_value) & (iju[1] < dim_value)
        cache[dim_value] = np.nonzero(mask)[0]
    return cache[dim_value]

def build_shared_draws(d_max, kappa, sigma, alpha, gamma, tau2, offset, a_mode, seed):
    rng = np.random.default_rng(seed)
    m_max = d_max * (d_max - 1) // 2
    prior_diag_max = prior_diag_from_powerlaw(d_max, alpha=alpha, gamma=gamma, tau2=tau2, offset=offset)
    if a_mode == 'nearest_neighbor':
        omegas = make_omegas_power(d_max, beta=alpha, c=2.0 ** (-gamma), offset=offset)
        A_true_max = make_Astar_nn(d_max, omegas)
        a_true_max = params_from_skew(A_true_max)
    elif a_mode == 'prior':
        z_prior = rng.standard_normal(m_max)
        a_true_max = z_prior * np.sqrt(prior_diag_max)
        A_true_max = make_Astar_from_atrue(d_max, a_true_max)
    else:
        raise ValueError("a_mode must be 'nearest_neighbor' or 'prior'")
    g_max = np.zeros(d_max, dtype=float)
    g_max[0] = 1.0
    theta_true_max = solve_theta(d_max, a_true_max, g_max, kappa)
    noise_max = rng.standard_normal(d_max)
    z_init = rng.standard_normal(m_max)
    a_init_max = z_init * np.sqrt(prior_diag_max)
    return {
        'd_max': d_max,
        'kappa': kappa,
        'sigma': sigma,
        'alpha': alpha,
        'gamma': gamma,
        'tau2': tau2,
        'offset': offset,
        'a_mode': a_mode,
        'param_iju': np.triu_indices(d_max, k=1),
        'param_indices_cache': {},
        'prior_diag': prior_diag_max,
        'a_true': a_true_max,
        'A_true': A_true_max,
        'g': g_max,
        'theta_true': theta_true_max,
        'noise': noise_max,
        'a_init': a_init_max,
    }

def generate_advection_diffusion_data_shared(dim_value, obs_indices, shared_draws):
    param_idx = get_param_indices_for_dim(dim_value, shared_draws)
    prior_diag = shared_draws['prior_diag'][param_idx]
    a_true = shared_draws['a_true'][param_idx]
    a_init = shared_draws['a_init'][param_idx]
    g = shared_draws['g'][:dim_value]
    theta_true = solve_theta(dim_value, a_true, g, shared_draws['kappa'])
    y_full = theta_true + shared_draws['sigma'] * shared_draws['noise'][:dim_value]
    y = y_full[obs_indices]
    return {
        'dim': dim_value,
        'obs_indices': np.asarray(obs_indices, dtype=int),
        'y': y,
        'y_full': y_full,
        'g': g,
        'a_true': a_true,
        'A_true': shared_draws['A_true'][:dim_value, :dim_value],
        'prior_diag': prior_diag,
        'theta_true': theta_true,
        'a_init': a_init,
    }

In [ ]:
obs_indices = get_obs_indices(d, obs_highest_freq, obs_bandwidth)
shared_draws = build_shared_draws(d, kappa, sigma, alpha, gamma, tau2, 1.0, a_mode, seed_data)
data = generate_advection_diffusion_data_shared(d, obs_indices, shared_draws)
problem = AdvectionDiffusionToy(
    dim=d,
    kappa=kappa,
    sigma=sigma,
    y=data['y'],
    obs_indices=data['obs_indices'],
    g=data['g'],
    prior_diag=data['prior_diag'],
)
x0 = np.array(data['a_init'], copy=True)

data_id_config = {'seed_data': seed_data, 'config_id': config_id, 'd': d, 'obs_highest_freq': obs_highest_freq, 'obs_bandwidth': obs_bandwidth, 'kappa': kappa, 'sigma': sigma, 'alpha': alpha, 'gamma': gamma, 'tau2': tau2, 'a_mode': a_mode}
run_id_config = {'seed_mcmc': seed_mcmc, 'n_iters': n_iters, 'burn_in': burn_in, 'P': P, 'mess_M': mess_M}
data_hash = stable_hash(data_id_config)
run_hash = stable_hash({'data_hash': data_hash, **run_id_config})

estimations_dir = repo_root / 'estimations' / 'solute_transport_ess_mess' / f'data_{data_hash}' / 'fixed' / f'ess_mess_convergence_{run_hash}'
reports_dir = repo_root / 'reports' / 'solute_transport_ess_mess' / f'data_{data_hash}' / f'ess_mess_convergence_{run_hash}'
mess_chains_dir = estimations_dir / 'chains' / 'mess_independent'
ess_root_dir = estimations_dir / 'chains' / 'independent_chains'
metrics_dir = estimations_dir / 'metrics'
for p in [estimations_dir, reports_dir, mess_chains_dir, ess_root_dir, metrics_dir]:
    p.mkdir(parents=True, exist_ok=True)

np.savez_compressed(estimations_dir / 'simulated_data.npz', y=data['y'], y_full=data['y_full'], a_true=data['a_true'], A_true=data['A_true'], prior_diag=data['prior_diag'], theta_true=data['theta_true'], a_init=data['a_init'], obs_indices=data['obs_indices'])

print('estimations_dir:', estimations_dir)
print('reports_dir:', reports_dir)

In [ ]:
def mess_chain_path(rep_idx):
    seed = seed_mcmc + 5000 + rep_idx
    return mess_chains_dir / f'mess_M{mess_M}_seed{seed}_chain{rep_idx:03d}.npz'

def ess_rep_dir(rep_idx):
    p = ess_root_dir / f'replicate_{rep_idx:03d}'
    p.mkdir(parents=True, exist_ok=True)
    return p

def ess_chain_path(rep_idx, chain_idx):
    return ess_rep_dir(rep_idx) / f'ess_independent_seed{seed_mcmc}_rep{rep_idx:03d}_chain{chain_idx:03d}.npz'

def run_chain_mess(x_init, n_steps, seed, M):
    rng = np.random.default_rng(seed)
    x = np.array(x_init, copy=True)
    chain = np.zeros((n_steps, x.shape[0]), dtype=float)
    intervals = np.zeros(n_steps, dtype=int)
    t0 = time.perf_counter()
    for t in range(n_steps):
        x, n_intervals, _ = mess_step(x, problem, rng, M=M, use_lp=False)
        chain[t] = x
        intervals[t] = n_intervals
    runtime = time.perf_counter() - t0
    return chain, intervals, runtime

def load_chain(path):
    if not path.exists():
        return None
    z = np.load(path, allow_pickle=False)
    chain = z['chain']
    accept_rate = float(z['accept_rate']) if 'accept_rate' in z else np.nan
    runtime_sec = float(z['runtime_sec']) if 'runtime_sec' in z else np.nan
    z.close()
    return chain, accept_rate, runtime_sec

def ensure_mess_chain(rep_idx):
    path = mess_chain_path(rep_idx)
    if path.exists():
        return path, False
    seed = seed_mcmc + 5000 + rep_idx
    chain, intervals, runtime = run_chain_mess(x0, n_iters, seed, M=mess_M)
    np.savez_compressed(path, chain=chain, intervals=intervals, accept_rate=np.array(1.0), runtime_sec=np.array(runtime), rep_idx=np.array(rep_idx), M=np.array(mess_M))
    return path, True

def ensure_ess_chain(rep_idx, chain_idx):
    path = ess_chain_path(rep_idx, chain_idx)
    if path.exists():
        return path, False
    seed = seed_mcmc + 100000 + rep_idx * 1000 + chain_idx
    chain, intervals, runtime = run_chain_mess(x0, n_iters, seed, M=1)
    np.savez_compressed(path, chain=chain, intervals=intervals, accept_rate=np.array(1.0), runtime_sec=np.array(runtime), rep_idx=np.array(rep_idx), chain_idx=np.array(chain_idx))
    return path, True

In [ ]:
created_mess = 0
created_ess = 0
for rep_idx in range(target_replicates):
    _, made_mess = ensure_mess_chain(rep_idx)
    created_mess += int(made_mess)
    for chain_idx in range(P):
        _, made_ess = ensure_ess_chain(rep_idx, chain_idx)
        created_ess += int(made_ess)
print(f'Created {created_mess} new MESS chains and {created_ess} new ESS chains.')

In [ ]:
mess_chains = []
ess_groups = []
for rep_idx in range(target_replicates):
    mess_loaded = load_chain(mess_chain_path(rep_idx))
    if mess_loaded is None:
        continue
    mess_chains.append(mess_loaded[0])
    group = []
    for chain_idx in range(P):
        loaded = load_chain(ess_chain_path(rep_idx, chain_idx))
        if loaded is None:
            group = []
            break
        group.append(loaded[0])
    if len(group) == P:
        ess_groups.append(group)

if not mess_chains:
    raise ValueError('No MESS chains found.')
if not ess_groups:
    raise ValueError('No full ESS replicate groups found.')

ess_chains = [group[0] for group in ess_groups]
print('Loaded MESS chains:', len(mess_chains))
print('Loaded ESS replicate groups:', len(ess_groups))
print('Loaded ESS single-chain set:', len(ess_chains))

In [ ]:
parameter_observables = make_parameter_observables(problem, d)
parameter_observables = update_parameter_observables(parameter_observables)
parameter_observables = add_first_row_sum_observable(parameter_observables, d)
A_observables_raw = make_antisymmetric_A_observables(k_band=A_band_k, top_k=A_top_k, eps=A_support_eps, row_index=A_row_index, full_row=A_use_full_row)
A_observables_raw = [obs for obs in A_observables_raw if obs.obs_id != 104]
A_observables_raw = update_A_observables(A_observables_raw)
A_observables = wrap_A_observables_for_params(A_observables_raw, d, make_Astar_from_atrue)
observable_defs = parameter_observables + A_observables
if trace_observable_ids is None:
    trace_plot_defs = observable_defs
else:
    trace_plot_defs = select_observables(observable_defs, trace_observable_ids)
if mse_observable_ids is None:
    mse_target_observables = observable_defs
else:
    mse_target_observables = select_observables(observable_defs, mse_observable_ids)
if not mse_target_observables:
    raise ValueError('No observables selected for MSE plots.')
print('Observable count:', len(observable_defs))

In [ ]:
target_cache_path = metrics_dir / (
    f'pub_observables_true_mean_long_mess_M{long_target_mess_M}_'
    f'iters{long_target_n_iters}_burn{long_target_burnin}.npz'
)

cached_observable_targets = {}
if true_observable_cache_mode != 'all':
    cached_observable_targets = load_cached_observable_targets(
        target_cache_path, observable_defs, long_target_burnin
    )

external_target_path = None
if prefer_mpcn_target_cache:
    mpcn_exact = mcmc_repo_root / (
        'estimations/solute_transport/data_h4afe80f670cc/fixed/'
        'mpcn_pcn_convergence_h57eaaa0da6e8/metrics/'
        'pub_observables_true_mean_rho0p90000_burn5000.npz'
    )
    mpcn_candidates = [mpcn_exact]
    mpcn_candidates.extend(
        sorted(
            (mcmc_repo_root / 'estimations' / 'solute_transport').glob(
                'data_*/fixed/*/metrics/pub_observables_true_mean_rho*_burn5000.npz'
            )
        )
    )
    for candidate in mpcn_candidates:
        if not candidate.exists():
            continue
        ext_targets = load_cached_observable_targets(candidate, observable_defs, 5000)
        if ext_targets:
            external_target_path = candidate
            for name, value in ext_targets.items():
                cached_observable_targets.setdefault(name, value)
            break

forced_ids = set(true_observable_force_ids)
forced_names = {obs.name for obs in observable_defs if obs.obs_id in forced_ids}
missing = []
for obs in observable_defs:
    if true_observable_cache_mode == 'all':
        missing.append(obs)
    elif obs.name in forced_names:
        missing.append(obs)
    elif obs.name not in cached_observable_targets:
        missing.append(obs)

if true_observable_cache_mode == 'load' and missing:
    raise ValueError('Cache mode load-only but missing targets: ' + ', '.join(obs.name for obs in missing))

true_observable_targets = dict(cached_observable_targets)

if missing:
    long_chain_seed = seed_mcmc + 900000
    long_chain_path = metrics_dir / (
        f'long_target_mess_chain_M{long_target_mess_M}_'
        f'iters{long_target_n_iters}_seed{long_chain_seed}.npz'
    )

    if long_chain_path.exists():
        long_chain = load_chain(long_chain_path)[0]
    else:
        long_chain, long_intervals, long_runtime = run_chain_mess(
            x0,
            long_target_n_iters,
            long_chain_seed,
            M=long_target_mess_M,
        )
        np.savez_compressed(
            long_chain_path,
            chain=long_chain,
            intervals=long_intervals,
            accept_rate=np.array(1.0),
            runtime_sec=np.array(long_runtime),
            source=np.array('long_mess_target_chain'),
            burnin=np.array(long_target_burnin),
        )

    for obs in missing:
        true_observable_targets[obs.name] = compute_single_observable_target(
            [long_chain], obs, long_target_burnin
        )

    save_observable_target_cache(
        target_cache_path,
        observable_defs,
        true_observable_targets,
        long_target_burnin,
        1,
    )

observable_targets = {
    obs.name: float(true_observable_targets[obs.name])
    for obs in observable_defs
}

if external_target_path is not None:
    print('Loaded external target cache:', external_target_path)
print('Local target cache path:', target_cache_path)

In [ ]:
nr_replicates = min(len(mess_chains), len(ess_groups))
if nr_replicates < 1:
    raise ValueError('Need at least one replicate for MSE.')
effective_P = P
iter_grid = np.arange(1, n_iters + 1)

target_signature = stable_hash({obs.name: observable_targets[obs.name] for obs in observable_defs})
mse_cache_path = metrics_dir / (
    f'solute_transport_mse_observables_{rho_tag}_P{effective_P}_'
    f'reps{nr_replicates}_iters{n_iters}_target{target_signature}.npz'
)
mse_cached_results = load_cached_mse_results(mse_cache_path, n_iters, nr_replicates, effective_P)

def running_mean(x):
    x = np.asarray(x, dtype=float)
    return np.cumsum(x) / np.arange(1, x.shape[0] + 1, dtype=float)

mse_results_all = dict(mse_cached_results)
for obs in mse_target_observables:
    if obs.name in mse_results_all:
        continue
    target = observable_targets[obs.name]
    mess_running = []
    ess_running = []
    ep_ess_running = []
    for rep_idx in range(nr_replicates):
        mess_chain = mess_chains[rep_idx][:n_iters]
        ess_chain = ess_groups[rep_idx][0][:n_iters]
        ess_group = [ch[:n_iters] for ch in ess_groups[rep_idx][:effective_P]]

        mess_series = observable_series(mess_chain, obs, n_iters)
        ess_series = observable_series(ess_chain, obs, n_iters)

        mess_running.append(running_mean(mess_series))
        ess_running.append(running_mean(ess_series))

        group_series = np.stack([observable_series(ch, obs, n_iters) for ch in ess_group], axis=0)
        ep_iter_mean = np.mean(group_series, axis=0)
        ep_ess_running.append(running_mean(ep_iter_mean))

    mess_running = np.stack(mess_running, axis=0)
    ess_running = np.stack(ess_running, axis=0)
    ep_ess_running = np.stack(ep_ess_running, axis=0)

    # Keep legacy keys for compatibility with the shared plotting helper.
    mse_results_all[obs.name] = {
        'obs_id': obs.obs_id,
        'label': obs.label,
        'target': float(target),
        'mpcn_mse': np.mean((mess_running - target) ** 2, axis=0),
        'pcn_mse': np.mean((ess_running - target) ** 2, axis=0),
        'ep_mse': np.mean((ep_ess_running - target) ** 2, axis=0),
    }

save_mse_cache(
    mse_cache_path,
    mse_target_observables,
    mse_results_all,
    iter_grid,
    nr_replicates=nr_replicates,
    effective_P=effective_P,
    ep_group_count=nr_replicates,
    burnin_true_mean=long_target_burnin,
    posterior_mean_chain_count=len(mess_chains),
)
mse_results = {obs.name: mse_results_all[obs.name] for obs in mse_target_observables}
print('MSE cache path:', mse_cache_path)

In [ ]:
from multiproposal.plotting.solute_transport_convergence import plot_solute_transport_pub_traceplots

plot_solute_transport_pub_traceplots(
    mpcn_chains=mess_chains,
    pcn_chains=ess_chains,
    observable_defs=observable_defs,
    observable_targets=observable_targets,
    pub_plot_observable_ids=pub_plot_observable_ids,
    reports_dir=reports_dir,
    rho_tag=rho_tag,
    n_iters=n_iters,
    mpcn_panel_title='MESS',
    pcn_panel_title='ESS',
    output_tag='mess_vs_ess',
    show=True,
)

In [ ]:
from multiproposal.plotting.solute_transport_convergence import plot_solute_transport_running_mse

plot_solute_transport_running_mse(
    mse_results=mse_results,
    plot_defs=mse_target_observables,
    iter_grid=iter_grid,
    reports_dir=reports_dir,
    rho_tag=rho_tag,
    effective_P=effective_P,
    max_xlim=20,
    mpcn_label='MESS',
    pcn_label='ESS',
    ep_label=f'EP-ESS ($p$={effective_P})',
    output_tag='mess_vs_ess',
    show=True,
)